Read the CSV files from the Lakehouse 'Files' directory and save them as Delta tables.

In [ ]:
movies = spark.read.csv(
    "Files/movies.csv",
    header=True,
    inferSchema=True
)

links = spark.read.csv(
    "Files/links.csv",
    header=True,
    inferSchema=True
)

ratings = spark.read.csv(
    "Files/ratings.csv",
    header=True,
    inferSchema=True
)

# Save to Spark tables
movies.write.mode("overwrite").saveAsTable("movies_raw")
links.write.mode("overwrite").saveAsTable("links_raw")
ratings.write.mode("overwrite").saveAsTable("ratings_raw")


Filter for Animated Movies and write the data to the animated_movies table.

In [ ]:
from pyspark.sql.functions import col

animated_movies = spark.table("movies_raw")\
    .filter(col("genres").contains("Animation"))

animated_movies.write.mode("overwrite").saveAsTable("animated_movies")

Join animated_movies to the links file to get the tmdbId which we will use to get movie descriptions.

In [ ]:
animated = spark.table("animated_movies")

links = spark.table("links_raw")

joined = animated.join(links, "movieId")

tmdb_df = joined.select("movieId", "title", "tmdbId")\
    .dropna(subset=["tmdbId"])


Retrieve the API key for TMDB from the key vault. Replace **`<yourkeyvault>`** and **`<secretname>`** in the cell below before running it.
Call the tmdb api to get movie descriptions, add the descriptions to the Movelens data and write it to the animated_movies_enriched table. This is likely to take about 10 minutes.

In [ ]:
import requests
import pandas as pd

movies_pd = tmdb_df.toPandas()

# --- Retrieve the API key/token securely from Key Vault. Replace <yourkeyvault> with the name of your key vault, and <secretname> with your secrent name ---
API_KEY = notebookutils.credentials.getSecret("https://<yourkeyvault>.vault.azure.net/", "<secretname>")

def get_tmdb_overview(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{int(tmdb_id)}"
    params = {
        "api_key": API_KEY,
        "language": "en-US"
    }

    r = requests.get(url, params=params)

    if r.status_code == 200:
        return r.json().get("overview")
    return None

movies_pd["description"] = movies_pd["tmdbId"].apply(get_tmdb_overview)

desc_spark = spark.createDataFrame(movies_pd)

final_df = spark.table("animated_movies") \
    .join(desc_spark.select("movieId","description"),
          "movieId",
          "left") \
    .fillna({"description":""})

final_df.write.mode("overwrite") \
    .saveAsTable("animated_movies_enriched")

Set the endpoint URL for your embedding model.

In [ ]:
embedding_url = ""

Retrieve the API key for your Foundry endpoint from the key vault. Replace **`<yourkeyvault>`** and **`<secretname>`** in the cell below before running it.
Generate embeddings for the concatenated value of title, genres and description. Write the movie data with embeddings to the animated_movies_with_embeddings table. This step is likely to take around 10 minutes.

In [ ]:
import requests
from pyspark.sql.functions import udf, concat_ws
from pyspark.sql.types import ArrayType, FloatType

# --- Retrieve the API key/token securely from Key Vault. Replace <yourkeyvault> with the name of your key vault, and <secretname> with your secrent name ---
api_key = notebookutils.credentials.getSecret("https://<yourkeyvault>.vault.azure.net/", "<secretname>")

# --- Check that embedding_url is defined, or raise exception ---
try:
    embedding_url
except NameError:
    raise RuntimeError("embedding_url is not defined. Please define it before running this cell.")

# --- Define an embedding function with explicit error handling for failed response ---
def get_embedding(text):
    if text is None:
        return None
#    headers = {
#        "Authorization": f"Bearer {token}",
#        "Content-Type": "application/json"
#    }
    headers = {
        "api-key": api_key,
        "Content-Type": "application/json"
    }
    payload = {"input": text}
    try:
        resp = requests.post(embedding_url, headers=headers, json=payload, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            return data["data"][0]["embedding"]
        else:
            raise RuntimeError(f"Embedding request failed! Status: {resp.status_code}, Response: {resp.text}")
    except Exception as ex:
        raise RuntimeError(f"Embedding request error: {ex}")

# --- Register UDF ---
embedding_udf = udf(get_embedding, ArrayType(FloatType()))

# --- Prepare the column to embed (concatenate title, genres, description), NULL safe ---
df_to_embed = spark.table("animated_movies_enriched").fillna({"description": ""})
df_to_embed = df_to_embed.withColumn(
    "embedding_input",
    concat_ws(" ", "title", "genres", "description")
)

# --- Apply the embedding model ---
df_with_embeddings = df_to_embed.withColumn(
    "embedding", embedding_udf("embedding_input")
)

# --- Save the result as a table (so you can use the embeddings downstream) ---
df_with_embeddings.write.mode("overwrite").saveAsTable("animated_movies_with_embeddings")
